In [1]:
from src.data.preprocessing import preprocessing_legal_pt_voto_relatorio

PATH_TO_EXPLANATION_PT = "../data/explanations/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_rep02/"
PATH_TO_EVALUATION_PT = "../data/evaluation/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_rep02/"
PATH_TO_TEMPLATE = "../data/evaluation/survey_template.txt"

MAP_LABELS_IT = {"0": "Released", "1": "NotReleased"}
MAP_LABELS_EN = {"0": "negative", "1": "positive"}
MAP_LABELS_PT = {"0": "Preso", "1": "Solto"}

In [2]:
import os

os.makedirs(PATH_TO_EXPLANATION_PT, exist_ok=True)
os.makedirs(PATH_TO_EVALUATION_PT, exist_ok=True)

In [3]:
import glob

explanations_files = glob.glob(PATH_TO_EXPLANATION_PT + "*.json")
explanations_files

['../data/explanations/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_rep02/1222.json',
 '../data/explanations/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_rep02/162.json',
 '../data/explanations/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_rep02/171.json',
 '../data/explanations/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_rep02/1737.json',
 '../data/explanations/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_rep02/1882.json',
 '../data/explanations/STF_HC_Voto_Relatorio/best_model_DiffPool_20250903_135039_lr0.0001_hd_32_bs4_dec0.1_lk10.0_en0.01_rc0.1_ct0.1_bl0.1_rp0.1_l20.01_

In [4]:
import os
import json
from pathlib import Path

TEMPLATE_EXPLANATION = open(PATH_TO_TEMPLATE, "r").read().strip()


def process_explanation(target: Path, output_folder: Path):
    with open(target, "r") as fp:
        json_explanation = json.load(fp)
    print(json.dumps(json_explanation, indent=3))

    os.makedirs(output_folder, exist_ok=True)

    print("File:", target)
    doc_id = target.stem
    y_true = json_explanation.get('y_pred')
    y_pred = json_explanation.get('y_test')
    content = json_explanation.get('original_content')

    template = str(TEMPLATE_EXPLANATION)

    template = template.replace("@DOC_ID", str(doc_id))
    template = template.replace("@LABEL", MAP_LABELS_IT[str(y_true)])
    template = template.replace("@PREDICTION", MAP_LABELS_IT[str(y_pred)])
    template = template.replace("@TEXT", preprocessing_legal_pt_voto_relatorio(content))

    explanation_per_hypernode = json_explanation["explanation"]
    for hyper_node_explanation_id in list(explanation_per_hypernode)[:2]:
        template_explanation = str(template)  # Copy for this specific explanation

        print("Hyper node: " + hyper_node_explanation_id)
        explanation_content = explanation_per_hypernode[hyper_node_explanation_id]

        words_l0 = explanation_content["words_l0"]
        cg_methods = explanation_content["words_cg_methods"]

        llm_search = cg_methods["llm_search"]
        semantic_search_l0 = cg_methods["semantic_search_l0"]
        semantic_search_l1 = cg_methods["semantic_search_l1"]
        top_l0 = cg_methods["top_l0"]

        template_explanation = template_explanation.replace("@HYPERNODE_ID", hyper_node_explanation_id)
        template_explanation = template_explanation.replace("@WORDS_L0", ", ".join(words_l0))

        # Method 1
        template_explanation = template_explanation.replace("@LLM_SEARCH", ", ".join(llm_search))
        # Method 2
        template_explanation = template_explanation.replace("@SEMANTIC_SEARCH_L1", ", ".join(semantic_search_l1))
        # Method 3
        template_explanation = template_explanation.replace("@SEMANTIC_SEARCH_L0", ", ".join(semantic_search_l0))
        # Method 4
        template_explanation = template_explanation.replace("@TOP_L0", ", ".join(top_l0))

        output_file = Path(
            output_folder) / f"Explanation_Doc-{int(doc_id):03d}_Hypernode_{int(hyper_node_explanation_id):03d}.txt"
        with open(output_file, "w") as fp:
            fp.write(template_explanation)

    # Doc ID

    # Label
    # Prediction

    # Questions
    # 1. Is the prediction correct?


In [5]:
for explanation_file in explanations_files:
    process_explanation(Path(explanation_file), PATH_TO_EVALUATION_PT)

{
   "explanation": {
      "1": {
         "words_l0": {
            "expedi\u00e7\u00e3o": 0.31275469064712524,
            "objetivando": 0.24655622243881226,
            "desatendimento": 0.19876661896705627
         },
         "l1_to_l2_assignments": [
            0.5235286951065063,
            0.096282459795475,
            0.08278469741344452,
            0.0547669380903244,
            0.05427740886807442,
            0.04609604924917221,
            0.02752508409321308,
            0.019351890310645103,
            0.01577155850827694,
            0.014077169820666313,
            0.007852491922676563,
            0.006618657149374485,
            0.006069111172109842,
            0.00372677412815392,
            0.003480143379420042,
            0.003439603839069605,
            0.0032854077871888876,
            0.0032338981982320547,
            0.003173939185217023,
            0.003126099705696106,
            0.003029291285201907,
            0.0030188269447535276,
   